# Integer Linear Programming (ILP) Solutions

This notebook contains the Python (`PuLP 4.0`) code and verified solutions for all Integer Linear Programming, Mixed-Integer Linear Programming (MILP), and Sensitivity Analysis examples presented in the `Integer_Linear_Programming.tex` slide deck for **BUS 104**.

### 1. Simple Pure Integer Linear Program & Branch-and-Bound

**Problem Formulation:**
$$\max z = 3x_1 + 2x_2$$
subject to:
$$\begin{aligned}
2x_1 + x_2 &\le 7 \quad (C_1) \\
-x_1 + x_2 &\le 3 \quad (C_2) \\
x_1, x_2 &\in \mathbb{Z}_{\ge 0}
\end{aligned}$$

The cell below runs the exact `PuLP 4.0` code from **Section 1** of the slides (`Simple_Pure_ILP`).

In [1]:
import pulp

# 1. Initialize Pure ILP Model (PuLP 4.0 API)
prob = pulp.LpProblem("Simple_Pure_ILP", pulp.LpMaximize)

# 2. Integer Decision Variables: x1, x2 in Z_{>= 0}
x1 = prob.add_variable("x1", lowBound=0, cat=pulp.LpInteger)
x2 = prob.add_variable("x2", lowBound=0, cat=pulp.LpInteger)

# 3. Objective Function
prob += 3 * x1 + 2 * x2, "Total_Profit"

# 4. Resource Constraints
prob += 2 * x1 + x2 <= 7, "C1"
prob += -x1 + x2 <= 3, "C2"

# 5. Solve and inspect Result
res = prob.solve()
print(f"Status: {res.status.name}")
print(f"Optimal z* = {pulp.value(prob.objective):.2f}")
print(f"x1* = {x1.varValue}, x2* = {x2.varValue}")

Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - cbc /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/63ca4ed502ee4ca3beb8c92ecc62315f-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/63ca4ed502ee4ca3beb8c92ecc62315f-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 7 COLUMNS
At line 18 RHS
At line 21 BOUNDS
At line 24 ENDATA
Problem MODEL has 2 rows, 2 columns and 4 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -12.6667 - 0.00 seconds
Cgl0004I processed model has 2 rows, 2 columns (2 integer (0 of which binary)) and 4 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0012I Integer solution of -11 found by DiveCoefficient after 0 iterations and 0 nodes (0.00 seconds)
Cbc0012I Integer solution of -12 found by DiveCoefficient after 1 iterations and 0 nodes (0.00 sec

#### 1b. Comparing the Continuous LP Relaxation vs. Pure ILP (Why Naive Rounding Fails)

In the continuous LP relaxation ($x_1, x_2 \ge 0$), the optimal solution is $(x_1^*, x_2^*) = (4/3, 13/3) \approx (1.33, 4.33)$ with $z_{LP}^* = 12.67$. Rounding down to $(1, 4)$ yields $z = 11.00$ (suboptimal), while rounding to $(1, 5)$, $(2, 4)$, or $(2, 5)$ violates the constraints. Branch-and-Bound finds the true integer optimum $(2, 3)$ with $z^* = 12.00$.

In [2]:
import pulp

# Solve the Continuous LP Relaxation
lp_relax = pulp.LpProblem("LP_Relaxation", pulp.LpMaximize)
x1_c = lp_relax.add_variable("x1", lowBound=0, cat=pulp.LpContinuous)
x2_c = lp_relax.add_variable("x2", lowBound=0, cat=pulp.LpContinuous)

lp_relax += 3 * x1_c + 2 * x2_c, "Total_Profit"
lp_relax += 2 * x1_c + x2_c <= 7, "C1"
lp_relax += -x1_c + x2_c <= 3, "C2"

lp_relax.solve(pulp.COIN_CMD(msg=False))
print(f"LP Relaxation Optimum : x1* = {x1_c.varValue:.4f}, x2* = {x2_c.varValue:.4f}, z_LP* = {pulp.value(lp_relax.objective):.4f}")
print(f"Naive Rounded (1, 4)  : x1  = 1.0000, x2  = 4.0000, z     = {3*1 + 2*4:.4f} (Suboptimal)")
print(f"True Pure ILP Optimum : x1* = 2.0000, x2* = 3.0000, z_ILP*= {3*2 + 2*3:.4f} (Optimal)")

LP Relaxation Optimum : x1* = 1.3333, x2* = 4.3333, z_LP* = 12.6667
Naive Rounded (1, 4)  : x1  = 1.0000, x2  = 4.0000, z     = 11.0000 (Suboptimal)
True Pure ILP Optimum : x1* = 2.0000, x2* = 3.0000, z_ILP*= 12.0000 (Optimal)


### 2. The 0-1 Knapsack Problem

**Problem Context:**
A hiker can carry at most $W = 10\text{ kg}$ and must choose which items to pack ($x_i \in \{0, 1\}$):
- **Laptop:** Value $= \$40$, Weight $= 4\text{ kg}$ (Ratio $= \$10.00/\text{kg}$)
- **Gold:** Value $= \$50$, Weight $= 3\text{ kg}$ (Ratio $= \$16.67/\text{kg}$)
- **Silver:** Value $= \$30$, Weight $= 6\text{ kg}$ (Ratio $= \$5.00/\text{kg}$)
- **Books:** Value $= \$10$, Weight $= 5\text{ kg}$ (Ratio $= \$2.00/\text{kg}$)

The cell below runs the exact `PuLP 4.0` code from **Section 2** of the slides (`Knapsack_0_1`).

In [3]:
import pulp

items = ["Laptop", "Gold", "Silver", "Books"]
val = {"Laptop": 40, "Gold": 50, "Silver": 30, "Books": 10}
wt  = {"Laptop": 4,  "Gold": 3,  "Silver": 6,  "Books": 5}

prob = pulp.LpProblem("Knapsack_0_1", pulp.LpMaximize)

# Binary variables x[i] in {0, 1} using PuLP 4.0 API
x = prob.add_variable_dicts("x", items, cat=pulp.LpBinary)

# Objective & Weight Capacity Constraint
prob += pulp.lpSum(val[i] * x[i] for i in items), "Total_Value"
prob += pulp.lpSum(wt[i] * x[i] for i in items) <= 10, "Max_Weight"

res = prob.solve()
z_opt = pulp.value(prob.objective)
print(f"Status: {res.status.name}, Optimal Value = ${z_opt:.0f}")
for i in items:
    if x[i].varValue > 0.5:
        print(f"  Take {i}: weight={wt[i]} kg, value=${val[i]}")

Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - cbc /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/01bc575d3cff43e1b7128946c1172c86-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/01bc575d3cff43e1b7128946c1172c86-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 6 COLUMNS
At line 23 RHS
At line 25 BOUNDS
At line 30 ENDATA
Problem MODEL has 1 rows, 4 columns and 4 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -105 - 0.00 seconds
Cgl0004I processed model has 1 rows, 4 columns (4 integer (4 of which binary)) and 4 elements
Cutoff increment increased from 1e-05 to 9.9999
Cbc0038I Initial state - 1 integers unsatisfied sum - 0.5
Cbc0038I Solution found of -90
Cbc0038I Before mini branch and bound, 3 integers at bound fixed and 0 continuous
Cbc0038I Full problem 1 rows 4 colu

#### 2b. When Does the Greedy Heuristic Fail? (Counterexample from Slides)

Consider a knapsack with capacity $W = 6\text{ kg}$ and three items:
- **Item 1:** Value $= \$32$, Weight $= 4\text{ kg}$ (Ratio $= \$8.00/\text{kg}$)
- **Item 2:** Value $= \$22$, Weight $= 3\text{ kg}$ (Ratio $= \$7.33/\text{kg}$)
- **Item 3:** Value $= \$22$, Weight $= 3\text{ kg}$ (Ratio $= \$7.33/\text{kg}$)

The greedy heuristic selects **Item 1** first (highest ratio $\$8.00/\text{kg}$), leaving $2\text{ kg}$ of unused capacity and earning **$z_{\text{greedy}} = \$32$**. The 0-1 ILP solver selects **Items 2 and 3**, filling all $6\text{ kg}$ and earning **$z^* = \$44$**.

In [4]:
import pulp

items_ce = ["Item_1", "Item_2", "Item_3"]
val_ce = {"Item_1": 32, "Item_2": 22, "Item_3": 22}
wt_ce  = {"Item_1": 4,  "Item_2": 3,  "Item_3": 3}

prob_ce = pulp.LpProblem("Knapsack_Greedy_Counterexample", pulp.LpMaximize)
x_ce = prob_ce.add_variable_dicts("x", items_ce, cat=pulp.LpBinary)

prob_ce += pulp.lpSum(val_ce[i] * x_ce[i] for i in items_ce), "Total_Value"
prob_ce += pulp.lpSum(wt_ce[i] * x_ce[i] for i in items_ce) <= 6, "Capacity_6kg"

prob_ce.solve(pulp.COIN_CMD(msg=False))
print(f"Greedy Heuristic Choice : ['Item_1']          -> Weight = 4 kg, Value = $32")
print(f"Optimal 0-1 ILP Choice  : {[i for i in items_ce if x_ce[i].varValue > 0.5]} -> Weight = 6 kg, Value = ${pulp.value(prob_ce.objective):.0f}")

Greedy Heuristic Choice : ['Item_1']          -> Weight = 4 kg, Value = $32
Optimal 0-1 ILP Choice  : ['Item_2', 'Item_3'] -> Weight = 6 kg, Value = $44


### 3. 0-1 Knapsack with Logical Constraints

**Logical Conditions Added to the 4-Item Knapsack Problem:**
1. **Mutual Exclusion:** Cannot take both Laptop and Gold ($x_{\text{Laptop}} + x_{\text{Gold}} \le 1$).
2. **Implication (If-Then):** If Gold is taken, Books must also be taken ($x_{\text{Gold}} - x_{\text{Books}} \le 0$).

The cell below runs the exact `PuLP 4.0` code from **Section 3** of the slides (`Knapsack_Logical`).

In [5]:
import pulp

items = ["Laptop", "Gold", "Silver", "Books"]
val = {"Laptop": 40, "Gold": 50, "Silver": 30, "Books": 10}
wt  = {"Laptop": 4,  "Gold": 3,  "Silver": 6,  "Books": 5}

prob = pulp.LpProblem("Knapsack_Logical", pulp.LpMaximize)
x = prob.add_variable_dicts("x", items, cat=pulp.LpBinary)

prob += pulp.lpSum(val[i] * x[i] for i in items), "Total_Value"
prob += pulp.lpSum(wt[i] * x[i] for i in items) <= 10, "Max_Weight"

# Rule 1 (Mutual Exclusion): Cannot take both Laptop and Gold
prob += x["Laptop"] + x["Gold"] <= 1, "Mutually_Exclusive"

# Rule 2 (Implication): If Gold is taken, Books must be taken
prob += x["Gold"] - x["Books"] <= 0, "Gold_Implies_Books"

res = prob.solve()
z_opt = pulp.value(prob.objective)
print(f"Status: {res.status.name}, Optimal Value = ${z_opt:.0f}")
print("Selected:", [i for i in items if x[i].varValue > 0.5])

Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - cbc /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/63c09495dbd14fb9ada2597909adbafe-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/63c09495dbd14fb9ada2597909adbafe-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 8 COLUMNS
At line 29 RHS
At line 33 BOUNDS
At line 38 ENDATA
Problem MODEL has 3 rows, 4 columns and 8 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -70 - 0.00 seconds
Cgl0004I processed model has 3 rows, 4 columns (4 integer (4 of which binary)) and 8 elements
Cutoff increment increased from 1e-05 to 9.9999
Cbc0038I Initial state - 0 integers unsatisfied sum - 1.11022e-16
Cbc0038I Solution found of -70
Cbc0038I Before mini branch and bound, 4 integers at bound fixed and 0 continuous
Cbc0038I Mini branch and bou

### 4. Fixed Setup Cost Problem (Mixed-Integer Linear Program)

**Problem Formulation:**
$$\max Z = 5x_1 + 3x_2 - 10y$$
subject to:
$$\begin{aligned}
x_1 + x_2 &\le 10 \quad (\text{Resource A}) \\
2x_1 + x_2 &\le 16 \quad (\text{Resource B}) \\
x_2 &\le 7 \quad (\text{Resource C}) \\
x_1 + x_2 - 10y &\le 0 \quad (\text{Big-}M \text{ Linking Constraint, } M = 10) \\
x_1, x_2 &\ge 0, \quad y \in \{0, 1\}
\end{aligned}$$

The cell below runs the exact `PuLP 4.0` code from **Section 4** of the slides (`Fixed_Setup_Cost_MILP`).

In [6]:
import pulp

prob = pulp.LpProblem("Fixed_Setup_Cost_MILP", pulp.LpMaximize)

# Continuous production variables and binary setup variable
x1 = prob.add_variable("x1", lowBound=0, cat=pulp.LpContinuous)
x2 = prob.add_variable("x2", lowBound=0, cat=pulp.LpContinuous)
y  = prob.add_variable("y", cat=pulp.LpBinary)

# Objective: Gross Profit - Fixed Setup Cost
prob += 5 * x1 + 3 * x2 - 10 * y, "Net_Profit"

# Resource constraints + Big-M linking constraint (M = 10)
prob += x1 + x2 <= 10, "Resource_A"
prob += 2 * x1 + x2 <= 16, "Resource_B"
prob += x2 <= 7, "Resource_C"
prob += x1 + x2 - 10 * y <= 0, "Big_M_Link"

res = prob.solve()
z_opt = pulp.value(prob.objective)
print(f"Status: {res.status.name}, Net Profit Z* = ${z_opt:.2f}")
print(f"Activate y*={int(y.varValue)}, x1*={x1.varValue}, x2*={x2.varValue}")

Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - cbc /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/468840eb071b4124b293292a87f69a2e-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/468840eb071b4124b293292a87f69a2e-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 9 COLUMNS
At line 23 RHS
At line 28 BOUNDS
At line 30 ENDATA
Problem MODEL has 4 rows, 3 columns and 8 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -32 - 0.00 seconds
Cgl0003I 0 fixed, 0 tightened bounds, 2 strengthened rows, 0 substitutions
Cgl0004I processed model has 2 rows, 3 columns (1 integer (1 of which binary)) and 6 elements
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solution found of -32
Cbc0038I Relaxing continuous gives -32
Cbc0038I Before mini branch and bound, 1 integers at b

### 5. Minimum Purchase Quantity (MOQ) Problem

**Problem Formulation:**
A firm may place an order with a supplier ($y \in \{0, 1\}$) with a fixed setup fee of $\$12$, a single resource constraint ($2x_1 + x_2 \le 16$ assembly hours), and a **two-sided switching rule** ($6y \le x_1 + x_2 \le 16y$) enforcing a Minimum Order Quantity (MOQ) of $6$ total units whenever $y = 1$:
$$\max Z = 7x_1 + 5x_2 - 12y$$
subject to:
$$\begin{aligned}
2x_1 + x_2 &\le 16 \quad (\text{Assembly Hours}) \\
x_1 + x_2 - 6y &\ge 0 \quad (\text{MOQ Lower Switch: } x_1 + x_2 \ge 6 \text{ if } y = 1) \\
x_1 + x_2 - 16y &\le 0 \quad (\text{Big-}M \text{ Upper Switch: } x_1 + x_2 = 0 \text{ if } y = 0) \\
x_1, x_2 &\in \mathbb{Z}_{\ge 0}, \quad y \in \{0, 1\}
\end{aligned}$$

The cell below runs the exact `PuLP 4.0` code from **Section 5** of the slides (`Supplier_Order_MOQ`).

In [7]:
import pulp

prob = pulp.LpProblem("Supplier_Order_MOQ", pulp.LpMaximize)

# Integer order quantities x1, x2 and binary order indicator y
x1 = prob.add_variable("x1", lowBound=0, cat=pulp.LpInteger)
x2 = prob.add_variable("x2", lowBound=0, cat=pulp.LpInteger)
y  = prob.add_variable("y", cat=pulp.LpBinary)

prob += 7 * x1 + 5 * x2 - 12 * y, "Net_Profit"

# Single resource constraint
prob += 2 * x1 + x2 <= 16, "Assembly_Hours"

# Two-sided switching constraints: 6*y <= x1 + x2 <= 16*y
prob += x1 + x2 - 6 * y >= 0, "MOQ_Lower_Switch"
prob += x1 + x2 - 16 * y <= 0, "Big_M_Upper_Switch"

res = prob.solve()
z_opt = pulp.value(prob.objective)
print(f"Status: {res.status.name}, Profit Z* = ${z_opt:.2f}")
print(f"y*={y.varValue:.0f}, x1*={x1.varValue:.0f}, x2*={x2.varValue:.0f}")

Welcome to the CBC MILP Solver 
Version: 2.10.13 
Build Date: Mar 11 2026 

command line - cbc /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/b4097e3a844342db82d9315e7dff70b6-pulp.mps -timeMode elapsed -solve -printingOptions all -solution /var/folders/x2/y620t7fd5dl06n7bq9jsb9qw0000gp/T/b4097e3a844342db82d9315e7dff70b6-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 8 COLUMNS
At line 26 RHS
At line 30 BOUNDS
At line 34 ENDATA
Problem MODEL has 3 rows, 3 columns and 8 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is -68 - 0.00 seconds
Cgl0003I 0 fixed, 0 tightened bounds, 1 strengthened rows, 0 substitutions
Cgl0004I processed model has 3 rows, 3 columns (3 integer (1 of which binary)) and 9 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0038I Initial state - 0 integers unsatisfied sum - 0
Cbc0038I Solution found of -68
Cbc0038I Cleaned solution of -68
Cbc0038I Bef

### 8. Last-Mile Delivery Routing & The "Subtour Trap" (0-1 Traveling Salesperson Problem)

**Problem Context:**
An e-commerce delivery van starts at **Depot 1** and must visit 5 customer stops ($\{2, 3, 4, 5, 6\}$) across two city clusters separated by a river (**West Cluster $\{1, 2, 3\}$** and **East Cluster $\{4, 5, 6\}$**) and return to Depot 1 in a **single connected tour** at minimum total mileage:
- **Within West Cluster $\{1, 2, 3\}$:** $c_{12} = 4,\; c_{23} = 4,\; c_{13} = 5\text{ miles}$
- **Within East Cluster $\{4, 5, 6\}$:** $c_{45} = 4,\; c_{56} = 4,\; c_{46} = 5\text{ miles}$
- **Cross-River Bridges:** Bridge A ($c_{24} = 8\text{ miles}$) and Bridge B ($c_{36} = 9\text{ miles}$); all other cross-river links cost $12\text{–}16\text{ miles}$.

#### 8a. The Naive Degree-Constrained Model (Demonstrating the Subtour Trap)
If we only enforce **Out-Degree $= 1$** ($\sum_{j \ne i} x_{ij} = 1$) and **In-Degree $= 1$** ($\sum_{i \ne j} x_{ij} = 1$) at every stop without Subtour Elimination Constraints, the solver cheats by creating **two disconnected 3-stop loops** ($1 \to 3 \to 2 \to 1$ and $4 \to 6 \to 5 \to 4$) with $Z_{\text{naive}} = 26\text{ miles}$, never crossing the river!

In [8]:
import pulp

nodes = [1, 2, 3, 4, 5, 6]
dist = {
    (1,2): 4, (1,3): 5, (2,3): 4, (4,5): 4, (4,6): 5, (5,6): 4,
    (2,4): 8, (3,6): 9, (1,4): 14, (1,5): 16, (1,6): 14,
    (2,5): 12, (2,6): 13, (3,4): 13, (3,5): 12
}
c = {(i,j): dist.get((i,j), dist.get((j,i))) for i in nodes for j in nodes if i != j}

# Naive Model: Degree Constraints Only (No Subtour Elimination)
prob_naive = pulp.LpProblem("Last_Mile_TSP_Naive", pulp.LpMinimize)
x_naive = {(i,j): prob_naive.add_variable(f"x_{i}_{j}", cat=pulp.LpBinary) for i, j in c}

prob_naive += pulp.lpSum(c[i,j] * x_naive[i,j] for i, j in c), "Total_Miles"
for i in nodes:
    prob_naive += pulp.lpSum(x_naive[i,j] for j in nodes if j != i) == 1, f"Out_{i}"
    prob_naive += pulp.lpSum(x_naive[j,i] for j in nodes if j != i) == 1, f"In_{i}"

res_naive = prob_naive.solve(pulp.COIN_CMD(msg=False))
naive_arcs = [(i,j) for (i,j) in c if x_naive[i,j].varValue > 0.5]
print(f"Naive Degree-Only Status: {res_naive.status.name}, Mileage Z_naive = {pulp.value(prob_naive.objective):.0f} miles (INVALID - Disconnected Subtours!)")
print(f"Selected Directed Arcs: {naive_arcs}")
print("  Subtour 1 (West Loop): 1 -> 3 -> 2 -> 1 (13 miles)")
print("  Subtour 2 (East Loop): 4 -> 6 -> 5 -> 4 (13 miles)")

Naive Degree-Only Status: Optimal, Mileage Z_naive = 26 miles (INVALID - Disconnected Subtours!)
Selected Directed Arcs: [(1, 3), (2, 1), (3, 2), (4, 6), (5, 4), (6, 5)]
  Subtour 1 (West Loop): 1 -> 3 -> 2 -> 1 (13 miles)
  Subtour 2 (East Loop): 4 -> 6 -> 5 -> 4 (13 miles)


#### 8b. Breaking the Loop: Exact TSP with Subtour Elimination Constraints (SEC)

For **every** proper subset of stops $S \subset \{1, \dots, 6\}$ of size $2 \le |S| \le 5$ (i.e., 2-stop, 3-stop, 4-stop, and 5-stop subsets), a single connected tour can use **at most $|S| - 1$ internal arcs** inside $S$:
$$\sum_{i \in S} \sum_{j \in S, \; j \ne i} x_{ij} \le |S| - 1$$
Using `itertools.combinations(nodes, k)` for `k in range(2, len(nodes))` eliminates all $15 + 20 + 15 + 6 = 56$ proper subsets of sizes $k \in \{2, 3, 4, 5\}$ and finds the true optimal connected tour of **$Z^* = 34\text{ miles}$**.

In [9]:
from itertools import combinations
import pulp

prob_tsp = pulp.LpProblem("Last_Mile_TSP_Exact", pulp.LpMinimize)
x = {(i,j): prob_tsp.add_variable(f"x_{i}_{j}", cat=pulp.LpBinary) for i, j in c}

prob_tsp += pulp.lpSum(c[i,j] * x[i,j] for i, j in c), "Total_Miles"
for i in nodes:
    prob_tsp += pulp.lpSum(x[i,j] for j in nodes if j != i) == 1, f"Out_{i}"
    prob_tsp += pulp.lpSum(x[j,i] for j in nodes if j != i) == 1, f"In_{i}"

# Eliminate subtours on ALL subsets S of size k = 2, 3, 4, 5 (2 <= |S| <= n-1)
for k in range(2, len(nodes)):
    for S in combinations(nodes, k):
        prob_tsp += pulp.lpSum(x[i,j] for i in S for j in S if i != j) <= k - 1

res_tsp = prob_tsp.solve(pulp.COIN_CMD(msg=False))
opt_arcs = {i: j for (i,j) in c if x[i,j].varValue > 0.5}
tour = [1]
while len(tour) <= len(nodes):
    tour.append(opt_arcs[tour[-1]])

print(f"Status: {res_tsp.status.name}, Optimal Tour Mileage Z* = {pulp.value(prob_tsp.objective):.0f} miles")
print(f"Selected Directed Arcs: {[(i,j) for (i,j) in c if x[i,j].varValue > 0.5]}")
print("Connected Single Tour :", " -> ".join(map(str, tour)))

Status: Optimal, Optimal Tour Mileage Z* = 34 miles
Selected Directed Arcs: [(1, 2), (2, 4), (3, 1), (4, 5), (5, 6), (6, 3)]
Connected Single Tour : 1 -> 2 -> 4 -> 5 -> 6 -> 3 -> 1


### 9. Sports Analytics: Fantasy Lineup Optimization with Chemistry & Stacking Rules

**Problem Context:**
You are drafting a **5-player starting lineup** (2 Guards, 2 Forwards, 1 Center) from a pool of 8 NBA players under a **$\$50{,}000$ Salary Cap** ($\le 50.0\text{k}$) to maximize total projected fantasy points:
- **Guards (pick 2):** $P_1$ Luka ($52\text{ pts}, \$12.5\text{k}$, Team A), $P_2$ Kyrie ($42\text{ pts}, \$9.5\text{k}$, Team A), $P_3$ Alex ($28\text{ pts}, \$6.0\text{k}$, Team B)
- **Forwards (pick 2):** $P_4$ LeBron ($48\text{ pts}, \$11.5\text{k}$, Team B), $P_5$ Draymond ($34\text{ pts}, \$7.5\text{k}$, Team C), $P_6$ Rui ($26\text{ pts}, \$5.5\text{k}$, Team B)
- **Centers (pick 1):** $P_7$ Gafford ($36\text{ pts}, \$8.0\text{k}$, Team A), $P_8$ Gobert ($38\text{ pts}, \$8.5\text{k}$, Team D)

**Front-Office & Locker-Room Logical Rules:**
1. **Rule 1 — Locker-Room Feud (Mutual Exclusion):** Draymond ($P_5$) and Gobert ($P_8$) refuse to play together ($x_5 + x_8 \le 1$).
2. **Rule 2 — Alley-Oop Lob Duo (Conditional Stack):** Lob center Gafford ($P_7$) requires elite passer Luka ($P_1$) ($x_7 - x_1 \le 0$).
3. **Rule 3 — "Only One Basketball" Usage Clash:** At most 2 of the 3 ball-dominant stars Luka ($P_1$), Kyrie ($P_2$), and LeBron ($P_4$) ($x_1 + x_2 + x_4 \le 2$).
4. **Rule 4 — League Team-Stacking Cap:** At most 2 players from Team A ($x_1 + x_2 + x_7 \le 2$).

The cell below compares the **Naive "Paper All-Star" Lineup** (ignoring chemistry rules) against the **Chemistry-Compliant Optimal Lineup** from **Section 9** of the slides.

In [10]:
import pulp

players = ["P1_Luka", "P2_Kyrie", "P3_Alex", "P4_LeBron",
           "P5_Draymond", "P6_Rui", "P7_Gafford", "P8_Gobert"]
pts = {"P1_Luka": 52, "P2_Kyrie": 42, "P3_Alex": 28, "P4_LeBron": 48,
       "P5_Draymond": 34, "P6_Rui": 26, "P7_Gafford": 36, "P8_Gobert": 38}
sal = {"P1_Luka": 12.5, "P2_Kyrie": 9.5, "P3_Alex": 6.0, "P4_LeBron": 11.5,
       "P5_Draymond": 7.5, "P6_Rui": 5.5, "P7_Gafford": 8.0, "P8_Gobert": 8.5}

def solve_lineup(include_chemistry=True):
    prob = pulp.LpProblem("Fantasy_Lineup_ILP", pulp.LpMaximize)
    x = prob.add_variable_dicts("x", players, cat=pulp.LpBinary)
    prob += pulp.lpSum(pts[i] * x[i] for i in players), "Projected_Points"
    prob += pulp.lpSum(sal[i] * x[i] for i in players) <= 50.0, "Salary_Cap"
    # Position Quotas (2G, 2F, 1C)
    prob += x["P1_Luka"] + x["P2_Kyrie"] + x["P3_Alex"] == 2, "Two_Guards"
    prob += x["P4_LeBron"] + x["P5_Draymond"] + x["P6_Rui"] == 2, "Two_Forwards"
    prob += x["P7_Gafford"] + x["P8_Gobert"] == 1, "One_Center"
    if include_chemistry:
        prob += x["P5_Draymond"] + x["P8_Gobert"] <= 1, "Feud_Exclusion"
        prob += x["P7_Gafford"] - x["P1_Luka"] <= 0, "Lob_Duo_Stack"
        prob += x["P1_Luka"] + x["P2_Kyrie"] + x["P4_LeBron"] <= 2, "Usage_Clash"
        prob += x["P1_Luka"] + x["P2_Kyrie"] + x["P7_Gafford"] <= 2, "TeamA_Limit"
    res = prob.solve(pulp.COIN_CMD(msg=False))
    roster = [p for p in players if x[p].varValue > 0.5]
    return res.status.name, pulp.value(prob.objective), sum(sal[p] for p in roster), roster

st0, z0, s0, r0 = solve_lineup(include_chemistry=False)
print("=== 1. Naive \"Paper All-Star\" Lineup (Ignoring Chemistry Rules) ===")
print(f"Status: {st0} | Projected Points = {z0:.0f} pts | Salary Used = ${s0:.1f}k")
print(f"Roster: {r0}")
print("  -> VIOLATES Rule 1 (Feud): Drafts both Draymond and Gobert (x5 + x8 = 2 > 1)")
print("  -> VIOLATES Rule 3 (Usage Clash): Drafts all 3 ball-dominant stars (x1 + x2 + x4 = 3 > 2)\n")

st1, z1, s1, r1 = solve_lineup(include_chemistry=True)
print("=== 2. Chemistry-Compliant Optimal Lineup (All 4 Logic Rules Active) ===")
print(f"Status: {st1} | Optimal Points Z* = {z1:.0f} pts | Salary Used = ${s1:.1f}k")
print(f"Roster: {r1}")
for p in r1:
    print(f"  - {p}: {pts[p]} pts (${sal[p]:.1f}k)")

=== 1. Naive "Paper All-Star" Lineup (Ignoring Chemistry Rules) ===
Status: Optimal | Projected Points = 214 pts | Salary Used = $49.5k
Roster: ['P1_Luka', 'P2_Kyrie', 'P4_LeBron', 'P5_Draymond', 'P8_Gobert']
  -> VIOLATES Rule 1 (Feud): Drafts both Draymond and Gobert (x5 + x8 = 2 > 1)
  -> VIOLATES Rule 3 (Usage Clash): Drafts all 3 ball-dominant stars (x1 + x2 + x4 = 3 > 2)

=== 2. Chemistry-Compliant Optimal Lineup (All 4 Logic Rules Active) ===
Status: Optimal | Optimal Points Z* = 198 pts | Salary Used = $45.5k
Roster: ['P1_Luka', 'P3_Alex', 'P4_LeBron', 'P5_Draymond', 'P7_Gafford']
  - P1_Luka: 52 pts ($12.5k)
  - P3_Alex: 28 pts ($6.0k)
  - P4_LeBron: 48 pts ($11.5k)
  - P5_Draymond: 34 pts ($7.5k)
  - P7_Gafford: 36 pts ($8.0k)


### Q&A & Key Findings Across All 9 Models

- **Pure ILP vs. LP Relaxation (Example 1):** Enforcing integrality reduces the maximum objective value from $z_{LP}^* = 12.67$ at $(1.33, 4.33)$ to $z_{ILP}^* = 12.00$ at $(2, 3)$.
- **0-1 Knapsack & Logical Constraints (Examples 2–3):** Under a $10\text{ kg}$ limit, packing Laptop ($4\text{ kg}, \$40$) and Gold ($3\text{ kg}, \$50$) achieves the maximum value of $\$90$. Adding mutual exclusion and conditional inclusion shifts the optimum to Laptop and Silver ($z^* = \$70$).
- **Fixed Setup Cost & Minimum Order Quantity MILPs (Examples 4–5):** Using a tight Big-$M$ linking constraint ($x_1 + x_2 - 10y \le 0$) in Example 4 yields $Z^* = \$32.00$ at $(6.0, 4.0, 1)$. In the MOQ problem (Example 5), enforcing $6y \le x_1 + x_2 \le 16y$ yields $Z^* = \$68.00$ at $(0, 16, 1)$.
- **Fixed-Charge Transshipment MILP (Example 6):** Opening Warehouses 1 and 2 ($y_1^*=1, y_2^*=1, y_3^*=0$ and $x_{1T}^*=50, x_{2T}^*=30, x_{TD}^*=80$) minimizes total setup + shipping cost at $Z^* = \$650.00$.
- **Grid Store Location / Set Covering (Example 7):** Covering all 6 sectors of a $2 \times 3$ city grid requires a minimum of $Z^* = 2$ stores (Central Spine $\{2, 5\}$ or Opposite Corners $\{1, 6\}$ / $\{3, 4\}$).
- **Last-Mile Delivery Routing & Subtour Elimination (Example 8):** Degree constraints alone ($\text{In}=1, \text{Out}=1$) fall into the **Subtour Trap**, producing two disconnected 3-stop loops ($1 \to 3 \to 2 \to 1$ and $4 \to 6 \to 5 \to 4$, $26\text{ miles}$). Adding Subtour Elimination Constraints ($\sum_{i,j \in S} x_{ij} \le |S|-1$) enforces global connectivity and finds the true optimal 6-stop tour $1 \to 2 \to 4 \to 5 \to 6 \to 3 \to 1$ ($Z^* = 34\text{ miles}$).
- **Sports Analytics Fantasy Lineup Optimization (Example 9):** Under a $\$50.0\text{k}$ salary cap, imposing locker-room feud ($x_5 + x_8 \le 1$), lob-stack ($x_7 - x_1 \le 0$), star usage ($x_1 + x_2 + x_4 \le 2$), and team-stacking ($x_1 + x_2 + x_7 \le 2$) constraints shifts the lineup from the infeasible "Paper All-Star" five ($214\text{ pts}$) to the chemistry-optimal roster `[P1_Luka, P3_Alex, P4_LeBron, P5_Draymond, P7_Gafford]` ($Z^* = 198\text{ pts}, \$45.5\text{k}$).